<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/12-caching-memory/04-executor-memory-gc.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 12.4 — Executor memory, overhead, and GC: read the signal first

Local-mode caveat: one JVM plays every executor, so absolute memory
numbers won't match a real cluster — but the DIAGNOSTIC ROUTINE
(check GC Time, check which budget is exhausted) transfers directly.
This notebook practices reading the signal, not tuning production
values.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("12.4-executor-memory-gc")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# local[*] note: one JVM plays driver + all executors, so the Executors tab
# shows a single "driver" entry — the memory MODEL below is identical to a
# real cluster, only the topology is collapsed.

## The three separate budgets, printed side by side

`executor.memory` (the 12.1 heap), `memoryOverhead` (off-heap JVM/
native), and `pyspark.memory` (the Python worker budget) are
independent knobs — confirm none of them derive from another.

In [ ]:
for k in ["spark.executor.memory",
          "spark.executor.memoryOverhead",
          "spark.executor.pyspark.memory",
          "spark.executor.cores"]:
    try:
        print(f"{k:<32} = {spark.conf.get(k)}")
    except Exception:
        print(f"{k:<32} = <unset — uses a computed/runtime default>")

## GC Time on the Stages tab, via the REST API

Every task metric includes `jvmGcTime` alongside its duration. A task
where GC eats a large fraction of its time is "slow" for a JVM
reason, not a logic reason.

In [ ]:
import urllib.request, json as _json

def gc_report(stage_id):
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages/{stage_id}"
    detail = _json.load(urllib.request.urlopen(url))[0]
    for t in list(detail["tasks"].values())[:8]:
        m = t["taskMetrics"]
        dur = t["duration"] or 1
        gc_pct = 100 * m["jvmGcTime"] / dur
        print(f"task {t['taskId']:>4}: duration={dur:>6}ms  gc_time={m['jvmGcTime']:>5}ms "
              f"({gc_pct:4.1f}% of task)")

def last_stage_id():
    app_id = spark.sparkContext.applicationId
    stages = _json.load(urllib.request.urlopen(
        f"http://localhost:4040/api/v1/applications/{app_id}/stages?status=complete"))
    return max(s["stageId"] for s in stages)

## Provoke object churn: a Python UDF chain vs the native equivalent

Same arithmetic, two execution paths (9.0's distinction). Compare GC
time on each — the UDF path creates far more short-lived objects
crossing the JVM/Python boundary.

In [ ]:
from pyspark.sql.types import LongType

@F.udf(LongType())
def slow_plus(x):
    return x + 1

big = spark.range(3_000_000).withColumn("v", col("id") * 2)

print("NATIVE path:")
big.select((col("v") + 1).alias("v2")).write.format("noop").mode("overwrite").save()
gc_report(last_stage_id())

print("\nPYTHON UDF path:")
big.select(slow_plus(col("v")).alias("v2")).write.format("noop").mode("overwrite").save()
gc_report(last_stage_id())

## Cores vs memory: more concurrent tasks, same execution-memory pool

Local mode caveat: `local[*]` uses all cores already; instead, simulate
the effect by shrinking `spark.sql.shuffle.partitions` relative to
data size while conceptually more "slots" compete — connect back to
12.2's spill diagnosis for the real symptom this produces.

In [ ]:
print("cores available to this local session:", spark.sparkContext.defaultParallelism)
print("In cluster mode, doubling --executor-cores without raising")
print("--executor-memory means this many MORE concurrent tasks share the")
print("SAME execution-memory region from 12.1 -- the mechanism is identical")
print("to what made 12.2's spill demo worse with too few shuffle partitions:")
print("less room per concurrent consumer, more spill.")

In [ ]:
spark.stop()

## Exercises

1. Extend the UDF-vs-native GC comparison to three stages: native,
   Python UDF, Pandas UDF (Module 9.1). Rank them by GC Time
   percentage and explain the ranking using what you know about each
   one's object-creation pattern.
2. Build a job with a huge broadcast variable (e.g. a 50 MB Python
   dict broadcast via `sc.broadcast`) referenced inside a UDF. Compare
   GC Time to a version without the broadcast.
3. Using `gc_report()`, find whether GC time correlates with task
   duration outliers the same way spill did in 12.2 — is one task's
   GC time responsible for a stage's tail latency?
4. Write the diagnostic routine from the lesson as an actual function:
   given a stage id, return one of `"gc-bound"`, `"spill-bound"`
   (12.2), or `"neither"` based on thresholds you choose from the
   REST API fields.
5. Read `spark.executor.extraJavaOptions` (may be unset locally).
   Research what flag you'd add to explicitly request G1GC vs the
   default, and what `-XX:+PrintGCDetails`-style flag you'd add to get
   raw GC logs for a real cluster investigation.

In [ ]:
# your exercise code here